# Hedge Fund Time-Series Forecasting: Walkthrough & Mini-Demo

**Authors:** Le Van Hiep, Nguyen Duy Hieu, Vu Minh Hieu, Tran Trung Hieu **Topic:** Multi-Horizon Financial Alpha Prediction with LightGBM & Dynamic Embargo  

---

### 1. Key Engineering Highlights
1. **Strict Look-Ahead Bias Prevention**: Static Proxy Encoding on `feature_al` computed strictly prior to the validation threshold.
2. **Dynamic Embargo Validation**: Isolation buffer $\delta = H + 25$ preventing Target Overlap Leakage between training and validation.
3. **Pseudo-Wavelet & Multi-scale Kinematics**: Bandpass frequency filters (EMA spans 3, 10, 25) + 2nd-order acceleration for inflection detection.
4. **Cross-Sectional Normalization**: Neutralize systemic Market Beta via timestamp-level Z-score.
5. **Zero-Inflation Guard**: Automated post-training variance collapse auditing (`atol=1e-12`).

In [ ]:
import sys, os
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.helpers import seed_everything, setup_plotting_theme
from src.data.proxy_encoder import ProxyTargetEncoder
from src.features.engineering import engineer_features
from src.models.trainer import HorizonModelTrainer
from src.models.metrics import weighted_rmse_score, audit_forecast_distribution

seed_everything(42)
setup_plotting_theme()
print("Environment initialized successfully!")

### 2. Synthetic Market Simulation
We generate a micro time-series containing multiple assets, industry categories, and non-stationary regimes for quick end-to-end execution.

In [ ]:
n_steps = 4000
n_assets = 5

records = []
for a_idx in range(n_assets):
    code = f"ASSET_{a_idx:02d}"
    sub_code = f"SUB_{a_idx % 2}"
    sub_cat = f"CAT_{a_idx % 2}"
    
    t = np.linspace(0, 50, n_steps)
    latent_signal = np.sin(t * 0.5) * 5.0 + np.random.normal(0, 0.5, n_steps)
    
    for step_i in range(n_steps):
        sig = latent_signal[step_i]
        records.append({
            "id": f"{code}_{step_i}",
            "code": code,
            "sub_code": sub_code,
            "sub_category": sub_cat,
            "ts_index": step_i + 1,
            "horizon": 1,
            "feature_al": sig + np.random.normal(0, 0.2),
            "feature_am": sig * 0.8 + np.random.normal(0, 0.2),
            "feature_cg": sig * 0.5 + np.random.normal(0, 0.3),
            "feature_by": np.random.normal(0, 1.0),
            "feature_s": np.random.normal(0, 1.0),
            "feature_u": np.random.normal(0, 1.0),
            "feature_ac": np.random.normal(0, 1.0),
            "feature_ab": np.random.normal(0, 1.0),
            "feature_m": np.random.normal(0, 1.0),
            "feature_l": np.random.normal(0, 1.0),
            "weight": 1.0 + np.random.exponential(0.5),
            "y_target": sig * 1.2 + np.random.normal(0, 0.4),
        })

df_sim = pd.DataFrame(records)
print(f"Simulated dataset shape: {df_sim.shape}")

### 3. Proxy Target Encoding (Look-Ahead Bias Prevention)
Calculate prior categorical knowledge strictly before `val_threshold = 3500`.

In [ ]:
encoder = ProxyTargetEncoder(val_threshold=3500, proxy_col="feature_al", ts_col="ts_index")
encoder.fit(df_sim)
print("Precomputed Proxy Stats:", encoder.stats)

### 4. Hybrid Feature Engineering: Pseudo-Wavelets & 2nd-Order Kinematics

In [ ]:
df_engineered = engineer_features(df_sim, encoder.stats, hz=1, ts_col="ts_index")
sample_asset = df_engineered[df_engineered["code"] == "ASSET_00"].iloc[:300]

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(sample_asset["ts_index"], sample_asset["feature_al"], label="Raw feature_al", color="#58a6ff")
axes[0].legend()
axes[0].set_title("Raw Alpha Signal")

axes[1].plot(sample_asset["ts_index"], sample_asset["feature_al_wavelet_short"], label="Pseudo-Wavelet (EMA3 - EMA10)", color="#3fb950")
axes[1].axhline(0, color="#8b949e", linestyle="--")
axes[1].legend()
axes[1].set_title("Bandpass Pseudo-Wavelet Decomposition")

axes[2].plot(sample_asset["ts_index"], sample_asset["feature_al_accel"], label="Acceleration (2nd-Order Derivative)", color="#f85149")
axes[2].axhline(0, color="#8b949e", linestyle="--")
axes[2].legend()
axes[2].set_title("Kinematic Acceleration (Inflection Point Lead)")
plt.tight_layout()
plt.show()

### 5. Dynamic Embargo Split & Seed-Ensemble Training
Verifying that buffer $\delta = H + 25$ blocks Target Overlap Leakage.

In [ ]:
trainer = HorizonModelTrainer(
    lgb_params={
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "n_estimators": 100,
        "num_leaves": 31,
        "verbosity": -1,
        "n_jobs": -1,
    },
    seeds=[42, 2024],
    val_threshold=3500,
    early_stopping_rounds=20,
)

exclude = {"id", "code", "sub_code", "sub_category", "horizon", "ts_index", "weight", "y_target"}
feature_cols = [c for c in df_engineered.columns if c not in exclude]

fit_mask, val_mask = trainer.get_embargo_split(df_engineered, hz=1, ts_col="ts_index")
X_fit = df_engineered.loc[fit_mask, feature_cols]
y_fit = df_engineered.loc[fit_mask, "y_target"]
w_fit = df_engineered.loc[fit_mask, "weight"].astype(float)

X_val = df_engineered.loc[val_mask, feature_cols]
y_val = df_engineered.loc[val_mask, "y_target"]
w_val = df_engineered.loc[val_mask, "weight"].astype(float)

models = trainer.train_seed_ensemble(X_fit, y_fit, w_fit, X_val, y_val, w_val)
val_preds = trainer.ensemble_predict(models, X_val)
score = weighted_rmse_score(y_val, val_preds, w_val)
print(f"\nValidated Weighted RMSE Score: {score:.5f}")

# Distribution audit
audit_forecast_distribution(val_preds, horizon_name="Simulation Val")